# 모듈 4. SQL/Python Tool 최소 권한

이 노트북은 SQL Tool과 Python 실행 Tool에 최소 권한 원칙을 적용하는 쉬운 버전을 실습합니다.

## 학습 목표

- SQL Agent의 query checker만으로는 왜 부족한지 설명할 수 있다.
- SQL 실행 전에 `SELECT`만 허용하고 `DROP`, `DELETE`, `UPDATE`, `INSERT`를 차단한다.
- Python 실행 Tool에서 위험 키워드와 실행 시간을 제한한다.
- 보안 없는 A안과 보안 적용 B안을 비교한다.

## 왜 이 보안 기능을 배워야 할까?

SQL Tool과 Python Tool은 강력합니다. 그래서 실수나 공격이 곧 데이터 삭제, 파일 접근, 네트워크 호출, 비밀정보 유출로 이어질 수 있습니다.

모델에게 "위험한 쿼리를 만들지 마"라고 말하는 것만으로는 충분하지 않습니다. 실제 실행 함수가 최소 권한 정책을 강제해야 합니다.

In [ ]:
import sqlite3
import concurrent.futures


def create_demo_db() -> sqlite3.Connection:
    conn = sqlite3.connect(":memory:")
    conn.executescript(
        '''
        CREATE TABLE employees (
            id INTEGER PRIMARY KEY,
            name TEXT NOT NULL,
            salary INTEGER NOT NULL
        );
        INSERT INTO employees (name, salary) VALUES
            ('Alice', 7000),
            ('Bob', 6500),
            ('Carol', 7200);
        '''
    )
    return conn


malicious_sql = "SELECT * FROM employees; DROP TABLE employees;"
safe_sql = "SELECT name, salary FROM employees ORDER BY salary DESC"

## A안. SQL 문자열을 그대로 실행하기

`executescript`처럼 여러 문장을 실행할 수 있는 방식에 사용자가 만든 SQL을 그대로 넘기면, 조회 요청처럼 보이는 문자열 뒤에 파괴적 명령이 붙을 수 있습니다.

In [ ]:
def unsafe_sql_tool(sql: str) -> str:
    conn = create_demo_db()
    conn.executescript(sql)
    try:
        rows = conn.execute("SELECT COUNT(*) FROM employees").fetchone()
        return f"employees 테이블 존재, row_count={rows[0]}"
    except sqlite3.Error as exc:
        return f"문제 발생: {exc}"


print(unsafe_sql_tool(malicious_sql))

## B안. SELECT 전용 SQL Tool 만들기

강의용 쉬운 버전으로 금지 키워드와 다중 문장 여부를 검사합니다. 실무에서는 SQL 파서, DB 계정 권한 분리, 읽기 전용 커넥션, Row/Column level security를 함께 사용합니다.

In [ ]:
FORBIDDEN_SQL_KEYWORDS = ["drop", "delete", "update", "insert", "alter", "truncate", "create"]


def validate_select_only(sql: str) -> tuple[bool, str]:
    normalized = " ".join(sql.lower().split())
    if not normalized.startswith("select "):
        return False, "SELECT 문만 허용됩니다."
    if ";" in normalized.rstrip(";"):
        return False, "여러 SQL 문장을 한 번에 실행할 수 없습니다."
    if any(keyword in normalized for keyword in FORBIDDEN_SQL_KEYWORDS):
        return False, "쓰기/삭제/DDL 키워드는 허용되지 않습니다."
    return True, "허용"


def safe_sql_tool(sql: str) -> str:
    allowed, reason = validate_select_only(sql)
    if not allowed:
        return f"거부: {reason}"

    conn = create_demo_db()
    rows = conn.execute(sql).fetchall()
    return f"조회 결과: {rows}"


print("악성 SQL:", safe_sql_tool(malicious_sql))
print("정상 SQL:", safe_sql_tool(safe_sql))

## Python Tool A/B 테스트

Python 실행 Tool은 데이터 분석 Agent에서 유용하지만, 제한 없이 열어두면 파일 접근이나 비밀정보 접근으로 이어질 수 있습니다.

In [ ]:
SECRET_TOKEN = "demo-secret-token"


def unsafe_python_tool(code_text: str) -> str:
    namespace = {"SECRET_TOKEN": SECRET_TOKEN}
    exec(code_text, namespace)
    return str(namespace.get("answer", "answer 변수가 없습니다."))


print(unsafe_python_tool("answer = SECRET_TOKEN"))

In [ ]:
FORBIDDEN_PYTHON_PATTERNS = [
    "import ",
    "__",
    "open(",
    "os.",
    "subprocess",
    "socket",
    "eval(",
    "exec(",
    "SECRET_TOKEN",
]


def run_with_timeout(code_text: str, timeout_seconds: float = 1.0) -> str:
    def _run() -> str:
        namespace = {"answer": None}
        exec(code_text, {"__builtins__": {"len": len, "sum": sum, "range": range}}, namespace)
        return str(namespace.get("answer"))

    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:
        future = executor.submit(_run)
        return future.result(timeout=timeout_seconds)


def safe_python_tool(code_text: str) -> str:
    normalized = code_text.replace(" ", "")
    for pattern in FORBIDDEN_PYTHON_PATTERNS:
        if pattern.replace(" ", "") in normalized:
            return f"거부: 금지 패턴 감지 - {pattern}"
    try:
        return run_with_timeout(code_text)
    except concurrent.futures.TimeoutError:
        return "거부: 실행 시간이 너무 깁니다."
    except Exception as exc:
        return f"실행 오류: {exc}"


print("악성 코드:", safe_python_tool("answer = SECRET_TOKEN"))
print("정상 코드:", safe_python_tool("answer = sum(range(10))"))

## 비교 정리

- A안은 SQL 다중 문장과 Python 비밀정보 접근을 그대로 허용했습니다.
- B안은 SQL을 SELECT 전용으로 제한하고 Python의 금지 패턴, 제한된 builtins, timeout을 적용했습니다.
- 실무에서는 문자열 검사만 믿지 말고 읽기 전용 DB 계정, 컨테이너 격리, 네트워크 차단, 파일시스템 제한, 실행 로그를 함께 설계해야 합니다.

## 실습 과제

1. `SELECT *`를 금지하고 필요한 컬럼만 조회하도록 SQL 검사를 개선해보세요.
2. Python Tool에서 `while True` 같은 무한 루프를 더 안전하게 차단하는 방법을 찾아보세요.